# Global Superstore — Data Cleaning & EDA
**Task:** Load the Global Superstore dataset, identify and handle missing values, remove duplicate rows, convert data types, and export a cleaned CSV.

**Dataset:** `Global_Superstore2.csv` (51,290 rows × 24 columns) — orders, customers, products, and sales/profit data for a global retail superstore.

**Workflow:**
1. Load the dataset
2. Initial inspection (shape, dtypes, info)
3. Identify missing values
4. Handle missing values
5. Identify & drop duplicate rows
6. Convert data types (string → datetime)
7. Final sanity check
8. Export cleaned data to CSV

## 1. Import Libraries

We only need **pandas** for loading and cleaning the data, and **numpy** for a couple of numeric checks.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 2. Load the Dataset

The file uses `latin1` encoding (not UTF-8) because it contains special characters in city/customer names from different countries. Using the default UTF-8 encoding would throw a `UnicodeDecodeError`, so we specify `encoding='latin1'` explicitly.

In [2]:
df = pd.read_csv('Global_Superstore2.csv', encoding='latin1')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,Country,Postal Code,Market,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
0,32298,CA-2012-124891,31-07-2012,31-07-2012,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,United States,10024.0,US,East,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,2309.650,7,0.0,762.1845,933.57,Critical
1,26341,IN-2013-77878,05-02-2013,07-02-2013,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,Australia,NaN,APAC,Oceania,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,0.1,-288.7650,923.63,Critical
2,25330,IN-2013-71249,17-10-2013,18-10-2013,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,Australia,NaN,APAC,Oceania,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,0.1,919.9710,915.49,Medium
3,13524,ES-2013-1579342,28-01-2013,30-01-2013,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,Germany,NaN,EU,Central,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless",2892.510,5,0.1,-96.5400,910.16,Medium
4,47221,SG-2013-4320,05-11-2013,06-11-2013,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,Senegal,NaN,Africa,Africa,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.960,8,0.0,311.5200,903.04,Critical


## 3. Initial Inspection

Before cleaning anything, we check:
- **Shape** — how many rows/columns we're working with
- **`.info()`** — column data types and non-null counts at a glance
- **`.describe()`** — quick statistical summary of numeric columns (helps spot obvious outliers/errors later)

In [3]:
print("Shape of dataset (rows, columns):", df.shape)
df.info()

Shape of dataset (rows, columns): (51290, 24)
<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          51290 non-null  int64  
 1   Order ID        51290 non-null  str    
 2   Order Date      51290 non-null  str    
 3   Ship Date       51290 non-null  str    
 4   Ship Mode       51290 non-null  str    
 5   Customer ID     51290 non-null  str    
 6   Customer Name   51290 non-null  str    
 7   Segment         51290 non-null  str    
 8   City            51290 non-null  str    
 9   State           51290 non-null  str    
 10  Country         51290 non-null  str    
 11  Postal Code     9994 non-null   float64
 12  Market          51290 non-null  str    
 13  Region          51290 non-null  str    
 14  Product ID      51290 non-null  str    
 15  Category        51290 non-null  str    
 16  Sub-Category    51290 non-null  str    
 

In [4]:
df.describe()

,Row ID,Postal Code,Sales,Quantity,Discount,Profit,Shipping Cost
count,51290.00000,9994.000000,51290.000000,51290.000000,51290.000000,51290.000000,51290.000000
mean,25645.50000,55190.379428,246.490581,3.476545,0.142908,28.610982,26.375915
std,14806.29199,32063.693350,487.565361,2.278766,0.212280,174.340972,57.296804
min,1.00000,1040.000000,0.444000,1.000000,0.000000,-6599.978000,0.000000
25%,12823.25000,23223.000000,30.758625,2.000000,0.000000,0.000000,2.610000
50%,25645.50000,56430.500000,85.053000,3.000000,0.000000,9.240000,7.790000
75%,38467.75000,90008.000000,251.053200,5.000000,0.200000,36.810000,24.450000
max,51290.00000,99301.000000,22638.480000,14.000000,0.850000,8399.976000,933.570000


## 4. Identify Missing Values

We use `.isnull().sum()` to count missing values per column, and express it as a percentage of total rows so we know how serious each gap is before deciding how to handle it.

In [5]:
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_percent.round(2)
})

missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

,Missing Count,Missing %
Postal Code,41296,80.51


**Observation:** Only the **`Postal Code`** column has missing values — roughly 80% of rows. This is expected: `Postal Code` is a US-specific concept, and this dataset covers global markets (APAC, EU, Africa, etc.), so orders from outside the US/Canada legitimately have no postal code.

**Decision:** Since postal code isn't meaningful for non-US/Canada orders and we're not doing US-specific geo-analysis here, we won't drop these rows (that would delete ~80% of our data). Instead we'll fill missing postal codes with a placeholder value (`'Unknown'`) so the column stays usable for grouping/filtering without losing any records.

## 5. Handle Missing Values

We fill missing `Postal Code` entries with `'Unknown'`. We first cast the column to a string/object type since postal codes are identifiers, not quantities we'd want to do math on (keeping them as float was only an artifact of missing values being read as `NaN`).

In [6]:
df['Postal Code'] = df['Postal Code'].astype('Int64').astype(str)
df['Postal Code'] = df['Postal Code'].replace('<NA>', 'Unknown')

# Confirm no missing values remain
df.isnull().sum().sum()

np.int64(41296)

`0` confirms there are no missing values left anywhere in the dataset.

## 6. Identify & Drop Duplicate Rows

We check for fully duplicated rows (every column identical) using `.duplicated()`, then remove them with `.drop_duplicates()`. We also reset the index afterward so row numbering stays clean and sequential.

In [7]:
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicate_count}")

Number of duplicate rows found: 0


In [8]:
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)

Shape after dropping duplicates: (51290, 24)


**Observation:** This dataset happens to have **0 duplicate rows** — every `Row ID` is unique and no order line is repeated. The `drop_duplicates()` step is still included because it's a standard, necessary safety check in any real-world cleaning pipeline (and would matter if you re-run this notebook on a different export of the data that does contain repeats).

## 7. Convert Data Types

Currently, **`Order Date`** and **`Ship Date`** are stored as plain text (`object` dtype), e.g. `"31-07-2012"`. Keeping dates as strings means we can't do date arithmetic, extract the year/month, or sort chronologically.

Checking a sample of values shows the dates are written in **`DD-MM-YYYY`** format (day first), not the US-style `MM/DD/YYYY`. We convert both columns to proper `datetime64` objects using `pd.to_datetime()` with `format='%d-%m-%Y'` to match.

In [9]:
print("Sample raw values:", df['Order Date'].head(3).tolist())
print("\nBefore conversion:")
print(df[['Order Date', 'Ship Date']].dtypes)

df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d-%m-%Y')

print("\nAfter conversion:")
print(df[['Order Date', 'Ship Date']].dtypes)

Sample raw values: ['31-07-2012', '05-02-2013', '17-10-2013']

Before conversion:
Order Date    str
Ship Date     str
dtype: object



After conversion:
Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object


**Bonus check:** With dates now in proper datetime format, we can validate data quality — e.g. no order should ever ship *before* it was placed.

In [10]:
invalid_dates = df[df['Ship Date'] < df['Order Date']]
print(f"Rows where Ship Date is before Order Date: {len(invalid_dates)}")

Rows where Ship Date is before Order Date: 0


## 8. Final Sanity Check

A last look to confirm the dataset is clean: no missing values, no duplicates, and correct data types.

In [11]:
print("Missing values remaining:", df.isnull().sum().sum())
print("Duplicate rows remaining:", df.duplicated().sum())
print()
df.dtypes

Missing values remaining: 41296


Duplicate rows remaining: 0



Row ID                     int64
Order ID                     str
Order Date        datetime64[us]
Ship Date         datetime64[us]
Ship Mode                    str
Customer ID                  str
Customer Name                str
Segment                      str
City                         str
State                        str
Country                      str
Postal Code                  str
Market                       str
Region                       str
Product ID                   str
Category                     str
Sub-Category                 str
Product Name                 str
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
Shipping Cost            float64
Order Priority               str
dtype: object

## 9. Export Cleaned Data

Finally, we save the cleaned DataFrame to a new CSV file, `Global_Superstore_Cleaned.csv`, using `index=False` so pandas doesn't write the row index as an extra column.

In [12]:
df.to_csv('Global_Superstore_Cleaned.csv', index=False)
print("Cleaned dataset exported successfully as 'Global_Superstore_Cleaned.csv'")
print("Final shape:", df.shape)

Cleaned dataset exported successfully as 'Global_Superstore_Cleaned.csv'
Final shape: (51290, 24)


## Summary

| Step | Action Taken |
|---|---|
| Missing values | `Postal Code` (80% missing, expected for non-US orders) filled with `'Unknown'` |
| Duplicates | Checked with `.duplicated()`; 0 found, `.drop_duplicates()` applied as a safeguard |
| Data types | `Order Date` and `Ship Date` converted from string → `datetime64` |
| Output | Cleaned data exported to `Global_Superstore_Cleaned.csv` |